In [1]:
# %pip install google-cloud-bigquery
# https://www.rudderstack.com/guides/how-to-access-and-query-your-bigquery-data-using-python-and-r/

In [2]:
# %pip install db-dtypes pandas-gbq

In [3]:
# %pip install google-cloud-bigquery-storage

In [4]:
from google.cloud import bigquery
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt
import db_dtypes

In [5]:
client = bigquery.Client(project='first-project-321219')

query = """
With events as (
      SELECT  
      -- event_id 8, 11 have same session_id. if i use event_id, it cause duplicate session
            e1.id AS event_id,
            e1.session_id,
            e1.user_id,
            e1.created_at AS cart_created_at,
            --e1.event_type AS cart_event,
            e2.created_at AS purchase_created_at,
            --e2.event_type AS purchased_event,
            CASE WHEN e2.id IS NOT NULL THEN 1 ELSE 0 END AS purchase_label,
            e1.city,
            e1.state, 
            e1.browser,
            e1.traffic_source,
            u.age,
            u.gender,
            u.country

      FROM `bigquery-public-data.thelook_ecommerce.events` e1
      LEFT JOIN `bigquery-public-data.thelook_ecommerce.events` e2 on e1.user_id = e2.user_id 
                                                                  and e2.event_type =  'purchase' 
                                                                  and e2.created_at>e1.created_at 
                                                                  and TIMESTAMP_DIFF(e2.created_at, e1.created_at, DAY) <=7
      LEFT JOIN `bigquery-public-data.thelook_ecommerce.users` u on e1.user_id = u.id
      WHERE e1.event_type = 'cart'
            AND e1.created_at >='2024-01-01'
)


SELECT --event_id,
        DISTINCT
        session_id,
         user_id,
         city,
         state,
         browser,
         traffic_source,
         age,
         country,
         max(cart_created_at) as cart_created_date,
        -- CASE WHEN IFNULL(sum(purchase_label),0)>0 THEN 1 ELSE 0 END AS purchased_flag_1,
         MAX(purchase_label) AS purchased_flag  
         
FROM events

--where session_id = 'f8dc71e1-6091-45e9-92ed-d174ee90cd12'

GROUP BY 1,2,3,4,5,6,7,8

"""

df = client.query(query).to_dataframe()
print(df.head())

                             session_id  user_id       city         state  \
0  7e79f65b-39a1-4c8a-961c-da554a68e524    82840  Bogatynia  Dolnośląskie   
1  235821a3-1599-49a7-a570-76c1f4461e4c    33325  Bogatynia  Dolnośląskie   
2  b463c6a6-efd5-466e-8289-16d63a36bc24    19164  Bogatynia  Dolnośląskie   
3  a7ecac58-093f-4527-acc3-13b5336b3857    92059  Bogatynia  Dolnośląskie   
4  7d2130a9-6d58-4153-bcd9-4ee2c1de5984     7378  Bogatynia  Dolnośląskie   

   browser traffic_source  age country         cart_created_date  \
0   Chrome          Email   66  Poland 2025-12-25 02:33:59+00:00   
1   Safari        Adwords   33  Poland 2024-06-22 00:17:18+00:00   
2   Chrome          Email   53  Poland 2025-08-03 01:24:09+00:00   
3  Firefox        Adwords   12  Poland 2025-12-11 17:51:26+00:00   
4  Firefox        Adwords   57  Poland 2026-01-20 18:14:26+00:00   

   purchased_flag  
0               1  
1               1  
2               1  
3               1  
4               1  


In [6]:
# data exploration
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 192988 entries, 0 to 192987
Data columns (total 10 columns):
 #   Column             Non-Null Count   Dtype              
---  ------             --------------   -----              
 0   session_id         192988 non-null  object             
 1   user_id            119930 non-null  Int64              
 2   city               192988 non-null  object             
 3   state              192988 non-null  object             
 4   browser            192988 non-null  object             
 5   traffic_source     192988 non-null  object             
 6   age                119930 non-null  Int64              
 7   country            119930 non-null  object             
 8   cart_created_date  192988 non-null  datetime64[us, UTC]
 9   purchased_flag     192988 non-null  Int64              
dtypes: Int64(3), datetime64[us, UTC](1), object(6)
memory usage: 15.3+ MB


In [7]:
client = bigquery.Client(project='first-project-321219')

query = """

      SELECT  
            e1.event_type,
            MAX(e1.created_at)
      FROM `bigquery-public-data.thelook_ecommerce.events` e1
    
      WHERE e1.created_at >='2024-01-01'
    GROUP BY e1.event_type
    LIMIT 100

"""

test = client.query(query).to_dataframe()
print(test.head())

   event_type                              f0_
0        cart        2026-01-23 00:41:59+00:00
1     product        2026-01-23 00:39:04+00:00
2      cancel        2026-01-22 19:40:00+00:00
3  department        2026-01-23 00:36:32+00:00
4        home 2026-01-23 00:20:07.450450+00:00


In [8]:
df['cart_created_date'].max()
# it updates daily. It shows Timestamp('2026-01-03 00:36:35.719031+0000', tz='UTC') on 03/01/2026. however, latest purchase event is 2026-01-06 23:35:53.846390+00:00. a bit weird.

Timestamp('2026-01-23 00:41:59+0000', tz='UTC')

In [9]:
# as there's over 72,000 null values in age, country, I analysis the number of customers by purchased_flag.
df_age_notna = df[df['age'].notna()].groupby('purchased_flag')['session_id'].count()
df_age_notna_total = df[df['age'].notna()]['session_id'].count()
df_age_notna_pct = df_age_notna / df_age_notna_total * 100
df_age_notna_pct

purchased_flag
0     0.22263
1    99.77737
Name: session_id, dtype: float64

In [10]:
df_age_notna = df[df['age'].isna()].groupby('purchased_flag')['session_id'].count()
df_age_notna_total = df[df['age'].isna()]['session_id'].count()
df_age_notna_pct = df_age_notna / df_age_notna_total * 100
df_age_notna
# no user_id, age, country are always 0 in response flag. if I put it into the model, it will always predict non user_id is 0 in response_flag. 
# Therefore, I remove the null value and only focus on users who can login and have age, country details.
# 05/01/2025: Feedback: actually, we want to model to: if it's null in user_id, predict 0, isn't it? keep it for now. 

purchased_flag
0    73058
Name: session_id, dtype: int64

In [11]:
df.duplicated().sum()

np.int64(0)

In [12]:
df_city_count = df['city'].value_counts()
print((df_city_count == 1).sum())
# there's 7039 cities, and 879 only appears once. I will exclude it in the model


731


In [13]:
df['state'].nunique()
# I have checked country which has 16 unique values. It's fine to keep it and use one hot encoding

228

In [14]:
df['traffic_source'].nunique()

5

In [15]:
df['hour'] = df['cart_created_date'].dt.hour
df['day_of_week'] = df['cart_created_date'].dt.dayofweek
df.head()

,session_id,user_id,city,state,browser,traffic_source,age,country,cart_created_date,purchased_flag,hour,day_of_week
0,7e79f65b-39a1-4c8a-961c-da554a68e524,82840,Bogatynia,Dolnośląskie,Chrome,Email,66,Poland,2025-12-25 02:33:59+00:00,1,2,3
1,235821a3-1599-49a7-a570-76c1f4461e4c,33325,Bogatynia,Dolnośląskie,Safari,Adwords,33,Poland,2024-06-22 00:17:18+00:00,1,0,5
2,b463c6a6-efd5-466e-8289-16d63a36bc24,19164,Bogatynia,Dolnośląskie,Chrome,Email,53,Poland,2025-08-03 01:24:09+00:00,1,1,6
3,a7ecac58-093f-4527-acc3-13b5336b3857,92059,Bogatynia,Dolnośląskie,Firefox,Adwords,12,Poland,2025-12-11 17:51:26+00:00,1,17,3
4,7d2130a9-6d58-4153-bcd9-4ee2c1de5984,7378,Bogatynia,Dolnośląskie,Firefox,Adwords,57,Poland,2026-01-20 18:14:26+00:00,1,18,1


In [16]:
df['country'].value_counts() #when we do one-hot encoding, we will keep China as it's the biggest sample size Country.

country
China             40371
United States     26987
Brasil            17729
South Korea        6311
United Kingdom     5824
France             5808
Germany            4790
Spain              4750
Japan              2860
Australia          2668
Belgium            1460
Poland              335
Colombia             20
Austria              16
Deutschland           1
Name: count, dtype: int64

In [17]:
def load_data(dataframe, datefrom = None, dateto = None):
    
    if not isinstance(dataframe, pd.DataFrame):   #isinstance(): check specified object is in specificed type.
        print('Error: Input is not a DataFrame')
  

    try:
        'purchased_flag' in dataframe.columns 
    except Keyerror as e:
        print(f'Column Error: {e}')
        
    try:
        df[datefrom] = pd.to_datetime(df['cart_created_date'])
    except Exception:
        print(f'Date Conversion Error: {Exception}')

    return df

    

In [18]:
df_initial = load_data(df)

In [19]:
df_initial

,session_id,user_id,city,state,browser,traffic_source,age,country,cart_created_date,purchased_flag,hour,day_of_week,None
0,7e79f65b-39a1-4c8a-961c-da554a68e524,82840,Bogatynia,Dolnośląskie,Chrome,Email,66,Poland,2025-12-25 02:33:59+00:00,1,2,3,2025-12-25 02:33:59+00:00
1,235821a3-1599-49a7-a570-76c1f4461e4c,33325,Bogatynia,Dolnośląskie,Safari,Adwords,33,Poland,2024-06-22 00:17:18+00:00,1,0,5,2024-06-22 00:17:18+00:00
2,b463c6a6-efd5-466e-8289-16d63a36bc24,19164,Bogatynia,Dolnośląskie,Chrome,Email,53,Poland,2025-08-03 01:24:09+00:00,1,1,6,2025-08-03 01:24:09+00:00
3,a7ecac58-093f-4527-acc3-13b5336b3857,92059,Bogatynia,Dolnośląskie,Firefox,Adwords,12,Poland,2025-12-11 17:51:26+00:00,1,17,3,2025-12-11 17:51:26+00:00
4,7d2130a9-6d58-4153-bcd9-4ee2c1de5984,7378,Bogatynia,Dolnośląskie,Firefox,Adwords,57,Poland,2026-01-20 18:14:26+00:00,1,18,1,2026-01-20 18:14:26+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...
192983,aa1aad23-15ef-49f7-834e-b675cff64ec8,91166,Kamilche,Washington,Chrome,Adwords,20,United States,2025-07-09 14:08:54+00:00,1,14,2,2025-07-09 14:08:54+00:00
192984,aa586a18-6d8c-471e-972f-61cebe0c0f38,15880,Yelm,Washington,Chrome,Email,33,United States,2025-11-29 12:00:22+00:00,1,12,5,2025-11-29 12:00:22+00:00
192985,dd24ff7e-2e5a-4a9a-8039-7962f097d376,93945,null,Washington,Chrome,Email,29,United States,2025-08-10 21:15:00+00:00,1,21,6,2025-08-10 21:15:00+00:00
192986,0b4ad70b-681a-4a00-8bda-856193116218,44890,Spokane,Washington,IE,Email,63,United States,2024-10-04 22:34:28+00:00,1,22,4,2024-10-04 22:34:28+00:00


In [20]:
df_initial['country'].unique()

array(['Poland', 'Austria', 'Colombia', 'Deutschland', None, 'Australia',
       'Belgium', 'Brasil', 'China', 'France', 'Germany', 'Japan',
       'South Korea', 'Spain', 'United Kingdom', 'United States'],
      dtype=object)

In [21]:
df_test = df_initial.copy()
if df_test['country'] == 'España':
    df_test['country'] = 'Spain'
df_test['country'].unique()

ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

In [ ]:
df_purchase_1 = df[df['purchased_flag']==1].shape[0]
df_purchase_0 = df[df['purchased_flag']==0].shape[0]
df_purchase_pct = df_purchase_1 / df_purchase_0
print(f'purchased: {df_purchase_1}')
print(f'not_purchased: {df_purchase_0}')

In [ ]:
from collections import Counter
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import RandomOverSampler

def prepare_data(df_input, target = 'purchased_flag', random_state = 42):

    df=df_input.copy()
    df = df.dropna(axis = 0)
    
    df[target] = df[target].fillna(0).astype(int)
    df['user_id_flag'] = df['user_id'].notna().astype(int)
    df['age'] = df['age'].fillna(0).astype(int)
    
    cols = ['session_id', 'user_id', 'city', 'state', 'cart_created_date', 'purchased_flag']
    
    # it has a None column. if I create a list with None, it return error because it's a None type. not the column name is called Nome.
    # Therefore, I have to use list comprehension
    X = df[[c for c in df.columns if c not in cols and c is not None]]
    y = df[target]
    
    for col in X.columns:
        if X[col].dtype in ('Int64', 'int32', 'int64', 'float64'):
            X[col].fillna(0)
        elif X[col].dtype == 'object':
            X[col].fillna('Unknown')
        else:
            X[col].fillna('Unavailable')

    X = pd.get_dummies(X, dtype = int)
    X.columns = X.columns.astype(str)
            
    df_purchase_1 = df[df['purchased_flag']==1].shape[0]
    df_purchase_0 = df[df['purchased_flag']==0].shape[0]
    df_purchase_pct = df_purchase_0 / df_purchase_1

    if df_purchase_pct <=0.8:
    
        oversample = RandomOverSampler(sampling_strategy='minority', random_state = random_state)
        X_over, y_over = oversample.fit_resample(X, y)
        df['purchased_flag'] = y_over
        # df['age'] = df['age'].fillna(0)
        print(f'Initial purchased_flag: {Counter(y)}')
        print(f'Oversampleing purchased_flag: {Counter(y_over)}')

        df_final = pd.DataFrame(X_over)
        df_final[target] = y_over
    else:
        print(f'purchased_flag" {Counter(y)}')
        df_final = X.copy()
        df_final[target] = y

    return df_final
    
  

In [ ]:
df_prepare = prepare_data(df_initial)
df_prepare

In [ ]:
df_prepare['purchased_flag'].value_counts()

In [ ]:
# pip install catboost

In [ ]:
# pip install ipywidgets

In [ ]:
# jupyter nbextension enable --py widgetsnbextension

In [ ]:
# pip install imbalanced-learn

In [ ]:
#DO NOT USE

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from catboost import CatBoostClassifier


# def prepare_data_catboost(df_input, target = 'purchased_flag', random_state = 42):

#     df=df_input.copy()
    
#     df[target] = df[target].fillna(0).astype(int)
#     df['user_id_flag'] = df['user_id'].notna().astype(int)

    
#     cols = ['session_id', 'user_id', 'city', 'state', 'cart_created_date', 'purchased_flag', 'user_id_flag', 'age']

    
    
#     X = df[[c for c in df.columns if c not in cols and c is not None]].copy()
#     y = df[target]

#     cat_features = [i for i, col in enumerate(X.columns) if X[col].dtype == 'object']
    
#     for col in X.columns:
#         if X[col].dtype in ('Int64', 'int32', 'int64', 'float64'):
#             X[col] = X[col].fillna(0)
#         elif X[col].dtype == 'object':
#             X[col] = X[col].fillna('Unknown')
#         else:
#             X[col] = X[col].fillna('Unavailable')

#     # X = pd.get_dummies(X, dtype = int)
#     X.columns = X.columns.astype(str)
            
#     df_purchase_1 = df[df['purchased_flag']==1].shape[0]
#     df_purchase_0 = df[df['purchased_flag']==0].shape[0]
#     df_purchase_pct = df_purchase_0 / df_purchase_1

#     if df_purchase_pct <=0.7:
#         catboost_model_weights = CatBoostClassifier(
#             iterations = 1000,
#             learning_rate=0.1,
#             depth=6,
#             eval_metric='AUC',
#             random_seed = random_state,
#             class_weights=[1,3],
#             verbose=100)
      
#         X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = random_state)
#         x_resampled, y_resampled = smote.fit_resample(X_train, y_train)
#         # model = CatBoostClassifier(verbose = 0)
#         model.fit(X_resampled, y_resampled)
#         y_pred = model.predict(X_test)
#         print('Predictions:', y_pred)


#         #     12/01/2026: precision is 1, recall is 0.99. it seems too good to be truth. Thereore, I use get_feature_importance to understand more.
        
#         # when I include user_id_flag and age, the importance score is:
# #               feature  importance
# # 6    user_id_flag   51.774693
# # 2             age   47.149338
# # 3         country    0.726212
# # 4            hour    0.203973
# # 5     day_of_week    0.126768
# # 1  traffic_source    0.012143
# # 0         browser    0.006872

#         # It seems: when user_id_flag = 0, the model will learn that purchase_flag = 0 - Data leakage. same logic as age. I will exclude them in X
#         # feat_imp = catboost_model_weights.get_feature_importance()
#         # cols = X.columns

#         importance_df = pd.DataFrame({'feature': cols, 'importance': feat_imp})
#         importance_df = importance_df.sort_values(by = 'importance', ascending = False)

#         print(importance_df.head(10))
        
#         # y_pred_weights = catboost_model_weights.predict(X_test)

#         print('CatBoost: ')
#         print(classification_report(y_test, y_pred_weights)) #print precision, recall, f1-score, support
#         # Precision =  True Positives / (True Positives+False Positives) Of all predicted positives, how many were correct?
#         # Recall = (True Positives / (True Positives+False Negative) Of all actual positives, how many did we find?
#         # F1 score = (2 * Precision * Recall) / (Precision + Recall)
#         print('ROC AUC Score:', roc_auc_score(y_test, catboost_model_weights.predict_proba(X_test)[:,1]))
#         print(f'purchased_flag {Counter(y_pred_weights)}')
#         df_final = X_test.copy()
#         df_final[target] = y_pred_weights
#     else:
#         print(f'purchased_flag" {Counter(y)}')
#         df_final = X.copy()
#         df_final[target] = y

#     return df_final
    
  

In [ ]:
# cols = ['session_id', 'user_id', 'city', 'state', 'cart_created_date', 'purchased_flag']
    
# X = df[[c for c in df.columns if c not in cols and c is not None]]

# cat_features = [i for i, col in enumerate(X.columns) if X[col].dtype == 'object']
# cat_features

In [ ]:
    # for i, col in enumerate(X.columns):
    #     print({i}, {col})

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from catboost import CatBoostClassifier
from imblearn.over_sampling import SMOTE

def prepare_data_catboost_2(df_input, target = 'purchased_flag', random_state = 42):

    df=df_input.copy()
    
    df[target] = df[target].fillna(0).astype(int)
    
    cols = ['session_id', 'user_id', 'city', 'state', 'cart_created_date', 'purchased_flag', 'user_id_flag','age', 'country']

    
    X = df.drop(columns =[c for c in cols if c in df.columns]).copy()
    y = df[target]

    count_0 = Counter(y)[0]
    count_1 = Counter(y)[1]
    spw = count_0 / count_1 
    print(f'scale_pos_weight: {spw:.2f}')

    
    for col in X.columns:
        if not pd.api.types.is_numeric_dtype(X[col]):
            X[col] = X[col].astype(str).replace('nan', 'Unknown').fillna('Unknown') # convert datetime to string. CatBoost can't deal with datetime format. 
        else:
            X[col] = X[col].fillna(0)

    cat_features_idx = [i for i, col in enumerate(X.columns) if not pd.api.types.is_numeric_dtype(X[col])]

       
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = random_state)

    model = CatBoostClassifier(
            iterations = 1000,
            learning_rate=0.1,
            depth=6,
            eval_metric='AUC',
            random_seed = random_state,
            scale_pos_weight=spw,
            verbose=100,
            early_stopping_rounds = 50)

    model.fit(X_train, y_train, eval_set=(X_test, y_test), cat_features = cat_features_idx)

    y_pred = model.predict(X_test)
    y_prod = model.predict_proba(X_test)[:, 1]

    print('CatBoost Report:')
    print(classification_report(y_test, y_pred))
    print(f'ROC AUC Score: {roc_auc_score(y_test, y_prod):.4f}')

    feat_imp = model.get_feature_importance()
    imp_df = pd.DataFrame({'feature': X.columns, 'importance_score': feat_imp}).sort_values('importance_score', ascending=False)
    print(imp_df)
        
    df_final = X_test.copy()
    df_final[target] = y_pred


    return df_final
    
  

In [ ]:
df_prepare_catboost_2 = prepare_data_catboost_2(df_initial)
df_prepare_catboost_2

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn import metrics


def logistics_regression(df, target = 'purchased_flag', test_size = 0.2, random_state = 16):
    
    feature_cols = [c for c in df.columns if c != 'purchased_flag']
    X = df[feature_cols]
    y = df[target]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = test_size, random_state = random_state)

    logreg = LogisticRegression(random_state = random_state, max_iter = 1000)
    # max_tier: maximum of 1000 iterations to find the optimal set of coefficients (parameters). 

    # fit the model with data
    logreg.fit(X_train, y_train)

    y_pred = logreg.predict(X_test)

    # model evaluation by using confusion matrix

    confusion_matrix = metrics.confusion_matrix(y_test, y_pred)

    # plot Confusion Matrix chart
    class_names = [0,1]
    fig, ax = plt.subplots()
    tick_marks = np.arange(len(class_names))
    plt.xticks(tick_marks, class_names)
    plt.yticks(tick_marks, class_names)

    sns.heatmap(pd.DataFrame(confusion_matrix), annot = True, cmap = 'YlGnBu', fmt = 'g')
    plt.tight_layout()
    plt.title('Confusion Matrix')
    plt.ylabel('Actual label')
    plt.xlabel('Predicted label')
    plt.show()

    # print classifiction report
    print(classification_report(y_test, y_pred, target_names = ['not purchased', 'purchased']))

    # plot ROC curve
    y_pred_proba = logreg.predict_proba(X_test)[:,1] # the probability of positive label
    fpr, tpr, thresholds = metrics.roc_curve(y_test, y_pred_proba)
    auc = metrics.roc_auc_score(y_test, y_pred_proba)
    plt.plot(fpr, tpr, label='purchased flag is 1, auc='+str(auc))
    plt.plot([0,1], [1,0], 'r--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.show()

    coef_df = pd.DataFrame({
        'Feature': X.columns,
        'Coefficient': logreg.coef_[0]})

    coef_df = coef_df.sort_values(by = 'Coefficient', ascending = False)
    print(coef_df)

    return logreg, auc, y_test, y_pred_proba

    

In [ ]:
df_oversample_logreg = logistics_regression(df_prepare)
df_oversample_logreg

In [ ]:
df_prepare['user_id_flag'].info()

In [ ]:
print(df_prepare_catboost_2.columns.tolist())

In [ ]:
df_clean_catboost = df_prepare_catboost_2.drop(columns=[None]) 

df_catboost_encoding = pd.get_dummies(df_clean_catboost, drop_first=True)

df_catboost_logreg = logistics_regression(df_catboost_encoding)
df_catboost_logreg

# 14/01/2026: email Egor, ask the result, how to optimise. and class weight vs scale?

In [ ]:
# pip install --user xgboost

In [ ]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import accuracy_score, classification_report
import xgboost as xgb
def xgboost_classifier(df, random_state = 1, enable_categorical = True, nfold = 5):
    cols = ['user_id_flag','purchased_flag']
    X = df[[c for c in df_prepare.columns if c not in cols]].copy()
    y = df['purchased_flag']

    # split the data
    X_train, X_test, y_train, y_test = train_test_split(X, y, random_state = random_state, stratify = y)

    # create classification matrics
    dtrain_clf = xgb.DMatrix(X_train, y_train, enable_categorical = enable_categorical)
    dtest_clf = xgb.DMatrix(X_test, y_test, enable_categorical = enable_categorical)

    params = {"objective": "binary:logistic", 
              'tree_method': 'hist',
             'eval_metric': ['logloss', 'auc','error']
             }
    # Cross Validation
    n = 1000

    cv_results = xgb.cv(
        params,
        dtrain_clf,
        num_boost_round=n,
        nfold = nfold,
        metrics=['logloss', 'auc', 'error'],
        early_stopping_rounds = 10
    )

    best_n = cv_results.shape[0]
    print(f'Best iteration: {best_n}')

    # Training Model
    model = xgb.train(
            params = params,
            dtrain = dtrain_clf,
        num_boost_round = best_n,
        evals=[(dtrain_clf, 'train'), (dtest_clf, 'test')],
        verbose_eval = False
    )

    y_prob = model.predict(dtest_clf)
    y_pred = [1 if p > 0.5 else 0 for p in y_prob]

    print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
    print(classification_report(y_test, y_pred))
        

    # results.keys()

    # max_auc = results['test-auc-mean'].max()

    # print(f'Max Test AUC: {max_auc}')
    
    
    return model

In [ ]:
xgboost_classifier(df_prepare)